In [ ]:
persons_raw = session.table(f"{DB}.{RAW}.STREAM_T_PERSONS").filter(
    col("METADATA$ACTION") == "INSERT"
).drop("METADATA$ACTION", "METADATA$ISUPDATE", "METADATA$ROW_ID").cache_result()
print("Stream read complete")

In [ ]:
# All transformations in a single .select() — generates ONE query plan
indicator_cols = [
    "DO_NOT_RESUSCITATE_IND", "EDUCATIONAL_PARENT_IND", "GUARDIANSHIP_CONSENT_IND",
    "ADOPTION_NOT_APPROPRIATE_IND", "BIO_PARENTS_RETURN_IND", "BIRTH_CERTIFICATE_AVLBL_IND",
    "NO_PHONE_IND", "PET_OWNER_IND", "SMOKER_IND", "SSN_VALIDATION_IND",
    "RESTRICTED_IND", "ADOPTION_MATCH_FOUND_IND", "APPROXIMATE_DOB_IND",
    "APPROX_LAST_ADOPTED_DATE_IND"
]

desc_cols = ["FIRST_NAME", "LAST_NAME", "MIDDLE_NAME"]
user_cols = ["ADD_USER", "MOD_USER"]
trim_cols = ["UC_LAST_NAME", "UC_FIRST_NAME", "UC_MIDDLE_NAME",
             "SDX_LAST_NAME", "SDX_FIRST_NAME", "SDX_MIDDLE_NAME"]

# Build select expressions
all_cols = [c.name for c in persons_raw.schema.fields]
select_exprs = []

for c in all_cols:
    if c == "LOAD_TIMESTAMP":
        select_exprs.append(col(c).alias("RAW_LOAD_TIMESTAMP"))
    elif c in indicator_cols:
        select_exprs.append(
            coalesce(when(trim(col(c)) == lit(""), lit("N")).otherwise(trim(col(c))), lit("N")).alias(c)
        )
    elif c in desc_cols:
        select_exprs.append(
            coalesce(when(trim(col(c)) == lit(""), lit("NA")).otherwise(trim(col(c))), lit("NA")).alias(c)
        )
    elif c in user_cols:
        select_exprs.append(
            coalesce(when(trim(col(c)) == lit(""), lit("SYSTEM")).otherwise(trim(col(c))), lit("SYSTEM")).alias(c)
        )
    elif c in trim_cols:
        select_exprs.append(trim(col(c)).alias(c))
    else:
        select_exprs.append(col(c))

persons_clean = persons_raw.select(select_exprs)
print("Transformations applied (single select)")

In [ ]:
# Split valid/invalid
valid_persons = persons_clean.filter(
    col("PERSON_ID").is_not_null() &
    col("FIRST_NAME").is_not_null() &
    col("LAST_NAME").is_not_null()
)

invalid_persons = persons_clean.filter(
    col("PERSON_ID").is_null() |
    col("FIRST_NAME").is_null() |
    col("LAST_NAME").is_null()
).with_column(
    "ERROR_REASON",
    when(col("PERSON_ID").is_null(), lit("PERSON_ID_NULL"))
    .when(col("FIRST_NAME").is_null(), lit("FIRST_NAME_NULL"))
    .otherwise(lit("LAST_NAME_NULL"))
)
print("Valid/invalid split defined")

In [ ]:
# Checksum + timestamp in one chain, then write
checksum_cols = [
    ("PERSON_ID", "number"), ("LAST_NAME", "string"), ("FIRST_NAME", "string"),
    ("MIDDLE_NAME", "string"), ("CITIZENSHIP_CODE", "string"),
    ("PRIMARY_LANGUAGE_CODE", "string"), ("GENDER_CODE", "string"),
    ("DO_NOT_RESUSCITATE_IND", "string"), ("BIRTH_DATE", "timestamp"),
    ("DECEASED_DATE", "timestamp"), ("ADD_USER", "string"),
    ("MOD_USER", "string"), ("DELETED_DATE", "timestamp")
]

checksum_exprs = []
for col_name, col_type in checksum_cols:
    if col_type == "string":
        checksum_exprs.append(coalesce(col(col_name), lit("")))
    else:
        checksum_exprs.append(coalesce(col(col_name).cast("string"), lit("")))

persons_final = valid_persons.with_column(
    "CHECKSUM", sha2(concat_ws(lit("|"), *checksum_exprs), 256)
).with_column(
    "STAGING_LOADED_TIMESTAMP", current_timestamp()
)

persons_final.write.save_as_table(
    table_name=f"{DB}.{STG}.{STG_PERSONS}",
    mode="truncate"
)
print(f"PERSONS loaded into {STG}.{STG_PERSONS}")

In [ ]:
# Handle invalid records — count once, reuse
invalid_count = invalid_persons.count()

if invalid_count > 0:
    invalid_persons.select(
        lit("T_PERSONS").alias("TABLE_NAME"),
        col("ERROR_REASON"),
        col("SOURCE_FILE_NAME"),
        col("RAW_LOAD_TIMESTAMP"),
    ).write.save_as_table(
        table_name=f"{DB}.{STG}.{INVALID_RECORDS}",
        mode="append"
    )
    print(f"Invalid records saved: {invalid_count}")
else:
    print("No invalid records")

In [ ]:
# Audit + notification — use Python UUID instead of SQL round-trip
rows_processed = session.table(f"{DB}.{STG}.{STG_PERSONS}").count()
status = 'SUCCESS' if invalid_count == 0 else 'PARTIAL_SUCCESS'

session.call(
    f"{DB}.{AUDIT}.{SP_LOG_AUDIT}",
    str(uuid.uuid4()),
    'STG_PERSONS_LOAD', 'STAGING',
    datetime.now(), datetime.now(),
    rows_processed, invalid_count, status, None
)

session.call(
    f"{DB}.{UTIL}.{SP_SEND_PIPELINE_NOTIFICATION}",
    status, 'STG_PERSONS_LOAD', 'STAGING',
    f'PERSONS staging completed. Rows: {rows_processed}, Failed: {invalid_count}'
)
print("Audit log inserted and email sent")

In [ ]:
print(f"STG_PERSONS complete | Valid: {rows_processed} | Invalid: {invalid_count} | Status: {status}")